# 1. Explore the local HARTH data — Member 1

HARTH is already available in data/harth/. This notebook reads those CSV files, creates data audits and explores the signals. There is no download or extraction step. Install requirements.txt once and select that environment as the kernel before starting.

Uses the fixed UCI release: 22 people, two accelerometers, six axes at 50 Hz. Acceleration is in g. All 12 labels are retained.

[UCI dataset description](https://archive.ics.uci.edu/dataset/779/harth) · [Authors’ implementation](https://github.com/ntnu-ai-lab/harth-ml-experiments)

These cells have not been executed.

In [ ]:
from pathlib import Path
import os
import sys
# Find the project from the workspace, project folder, or notebooks folder.
ROOT = next((p for p in [Path.cwd(), Path.cwd() / 'harth_project', Path.cwd().parent]
             if (p / 'har_utils.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the IMLProj workspace or harth_project folder.')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from har_utils import *

### Audit each participant

Count readings, invalid values and irregular timestamps, then record activity counts. The resulting two tables let us inspect both data quality and label coverage.

In [ ]:
def audit_data():
    subjects, counts = [], []
    for path in subject_files():
        frame = read_subject(path)
        dt = frame.timestamp.diff().dt.total_seconds()
        subjects.append({'subject': path.stem, 'rows': len(frame),
                         'missing_sensor_values': int(frame[AXES].isna().sum().sum()),
                         'nonfinite_sensor_values': int((~np.isfinite(frame[AXES])).sum().sum()),
                         'duplicate_timestamps': int(frame.timestamp.duplicated().sum()),
                         'irregular_intervals': int(((dt < .019) | (dt > .021)).sum()),
                         'recorded_minutes': len(frame) / 50 / 60})
        for label, n in frame.label.value_counts().items():
            counts.append({'subject': path.stem, 'label': int(label),
                           'activity': ACTIVITIES[label], 'samples': int(n)})
    return pd.DataFrame(subjects), pd.DataFrame(counts)

## Audit the existing dataset

Read the subject CSV files already in data/harth/. The audit is recomputed when this cell runs. We do not interpolate missing readings or silently reorder timestamps.

In [ ]:
(ROOT / 'results').mkdir(exist_ok=True)
record_environment()
audit, counts = audit_data()
audit.to_csv(ROOT / 'results/subject_audit.csv', index=False)
counts.to_csv(ROOT / 'results/class_counts.csv', index=False)
display(audit)
print('Total readings:', audit.rows.sum())

In [ ]:
example = read_subject(subject_files()[0])
display(example.head())
display(pd.Series(ACTIVITIES, name='activity').rename_axis('label').to_frame())

## Class imbalance and participant coverage

Check whether each activity occurs in every participant. A high accuracy can hide poor recall for rare activities.

In [ ]:
totals = counts.groupby('activity')['samples'].sum().sort_values()
totals.plot.barh(figsize=(9, 5), title='Raw readings by activity')
plt.tight_layout(); plt.show()
coverage = counts.pivot(index='subject', columns='activity', values='samples').fillna(0)
display(coverage)
display((coverage > 0).sum().sort_values().to_frame('subjects containing activity'))

In [ ]:
snippet = example.iloc[:500]
t = np.arange(len(snippet)) / 50
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for ax, sensor in zip(axes, ['back', 'thigh']):
    for axis in 'xyz':
        ax.plot(t, snippet[f'{sensor}_{axis}'], label=axis)
    ax.set_ylabel(f'{sensor} (g)'); ax.legend(ncol=3)
axes[-1].set_xlabel('Seconds'); plt.tight_layout(); plt.show()

## Handoff to Member 2

Explain the class balance, participant coverage and timestamp audit. One row is a reading, not a training window. Subject IDs and timestamps are metadata, never predictors.

Notebook 02 reads the same local data/harth subject files. On another machine, copy the data/harth folder and project code. No in-memory variables need to be passed between notebooks.